# Prédiction de Toxicité avec Vraies Données (ClinTox)
Dans ce notebook, nous n'utilisons plus des données inventées.
Nous allons télécharger la base de données **ClinTox** : un dataset scientifique qui contient des médicaments approuvés par la FDA et des molécules qui ont échoué lors d'essais cliniques en raison de leur toxicité.

In [10]:
# Installation de pandas si vous ne l'avez pas (retirer le # pour exécuter)
# %pip install pandas

In [11]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

# 1. TÉLÉCHARGEMENT DES VRAIES DONNÉES (ClinTox)
print("Téléchargement des données (environ 1500 molécules)...")
url = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/clintox.csv.gz'
df = pd.read_csv(url)

print(f"Nombre total de lignes téléchargées : {len(df)}")
df.head() # Affiche les 5 premières lignes du tableau

Téléchargement des données (environ 1500 molécules)...
Nombre total de lignes téléchargées : 1484


,smiles,FDA_APPROVED,CT_TOX
0,*C(=O)[C@H](CCCCNC(=O)OCCOC)NC(=O)OCCOC,1,0
1,[C@@H]1([C@@H]([C@@H]([C@H]([C@@H]([C@@H]1Cl)C...,1,0
2,[C@H]([C@@H]([C@@H](C(=O)[O-])O)O)([C@H](C(=O)...,1,0
3,[H]/[NH+]=C(/C1=CC(=O)/C(=C\C=c2ccc(=C([NH3+])...,1,0
4,[H]/[NH+]=C(\N)/c1ccc(cc1)OCCCCCOc2ccc(cc2)/C(...,1,0


In [12]:
# 2. NETTOYAGE ET CONVERSION EN VECTEURS (Morgan Fingerprints)
# La colonne 'CT_TOX' indique la toxicité clinique (1 = Toxique, 0 = Non toxique)

# Initialisation du générateur d'empreintes moderne
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

X_list = []
y_list = []

print("Génération des empreintes chimiques (cela peut prendre 2 ou 3 secondes)...")
for index, row in df.iterrows():
    smiles = row['smiles']
    tox = row['CT_TOX']
    
    mol = Chem.MolFromSmiles(smiles)
    # Parfois les bases de données ont des erreurs. 
    # RDKit renvoie None si le texte SMILES est chimiquement impossible.
    if mol is not None:
        fp = mfpgen.GetFingerprint(mol)
        X_list.append(np.array(fp))
        y_list.append(tox)

X = np.array(X_list)
y = np.array(y_list)

print(f"Molécules chimiquement valides conservées : {len(X)}")
print(f"-> Dont {sum(y)} toxiques (1) et {len(y) - sum(y)} non toxiques (0)")

Génération des empreintes chimiques (cela peut prendre 2 ou 3 secondes)...


[15:37:06] Explicit valence for atom # 0 N, 4, is greater than permitted
[15:37:06] Can't kekulize mol.  Unkekulized atoms: 9
[15:37:07] Can't kekulize mol.  Unkekulized atoms: 4
[15:37:07] Can't kekulize mol.  Unkekulized atoms: 4


Molécules chimiquement valides conservées : 1480
-> Dont 112 toxiques (1) et 1368 non toxiques (0)


In [13]:
# 3. SÉPARATION ENTRAÎNEMENT / TEST
# On garde 20% des molécules pour le test final (qu'on cache au modèle pendant l'entraînement)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Taille des données d'entraînement : {X_train.shape}")
print(f"Taille des données de test : {X_test.shape}")

Taille des données d'entraînement : (1184, 2048)
Taille des données de test : (296, 2048)


In [14]:
# 4. CRÉATION DU MODÈLE KERAS
# Le modèle est un peu plus grand car le problème est plus complexe que le précédent.
model = keras.Sequential([
    keras.Input(shape=(2048,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3), # On éteint 30% des neurones pour éviter l'overfitting
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 128)               262272    
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_4 (Dense)             (None, 64)                8256      
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 270593 (1.03 MB)
Trainable params: 270593 (1.03 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [15]:
# 5. ENTRAÎNEMENT SUR LES VRAIES DONNÉES

# Astuce IA : Il y a très peu de molécules toxiques (moins de 10%) dans cette base de données.
# Si on ne fait rien, le réseau va apprendre à dire "Non toxique" à chaque fois pour avoir 90% de précision !
# On utilise class_weight pour forcer le réseau à pénaliser beaucoup plus les erreurs sur les molécules toxiques.
poids_toxique = (len(y) - sum(y)) / sum(y)
class_weights = {0: 1.0, 1: poids_toxique}

print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=20,           # 20 passages complets sur les données
    batch_size=32,       # Il traite les molécules par paquets de 32
    verbose=1,
    validation_data=(X_test, y_test),
    class_weight=class_weights
)
print("Entraînement terminé !")

Début de l'entraînement...
Epoch 1/20
37/37 [==============================] - 1s 7ms/step - loss: 1.2638 - accuracy: 0.5144 - val_loss: 0.6028 - val_accuracy: 0.7838
Epoch 2/20
37/37 [==============================] - 0s 5ms/step - loss: 0.8796 - accuracy: 0.8539 - val_loss: 0.4927 - val_accuracy: 0.8007
Epoch 3/20
37/37 [==============================] - 0s 5ms/step - loss: 0.4686 - accuracy: 0.8910 - val_loss: 0.3698 - val_accuracy: 0.8615
Epoch 4/20
37/37 [==============================] - 0s 5ms/step - loss: 0.2808 - accuracy: 0.9485 - val_loss: 0.3687 - val_accuracy: 0.8851
Epoch 5/20
37/37 [==============================] - 0s 5ms/step - loss: 0.1851 - accuracy: 0.9704 - val_loss: 0.3598 - val_accuracy: 0.9088
Epoch 6/20
37/37 [==============================] - 0s 5ms/step - loss: 0.1433 - accuracy: 0.9772 - val_loss: 0.3658 - val_accuracy: 0.9088
Epoch 7/20
37/37 [==============================] - 0s 5ms/step - loss: 0.1635 - accuracy: 0.9747 - val_loss: 0.4529 - val_accuracy: 

In [ ]:
# 6. ÉVALUATION FINALE
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision réelle du réseau sur les {len(X_test)} molécules jamais vues : {accuracy * 100:.2f}%\n")

# 7. TEST SUR UN MÉDICAMENT RETIRÉ DU MARCHÉ
# Testons avec le Vioxx (Rofecoxib), un célèbre anti-inflammatoire 
# qui a été retiré du marché car il provoquait des crises cardiaques.
molecule_test_smiles = "CS(=O)(=O)c1ccc(cc1)C2=C(C(=O)OC2)c3ccccc3" 

mol = Chem.MolFromSmiles(molecule_test_smiles)
fp_test = mfpgen.GetFingerprint(mol)
X_nouveau = np.array(fp_test).reshape(1, -1)

probabilite = model.predict(X_nouveau, verbose=0)[0][0]

print(f"--- Analyse du Vioxx (Rofecoxib) ---")
if probabilite >= 0.5:
    print(f"Prédiction IA : TOXIQUE (Risque estimé à {probabilite*100:.2f}%)")
else:
    print(f"Prédiction IA : NON TOXIQUE (Sécurité estimée à {(1-probabilite)*100:.2f}%)")

Précision réelle du réseau sur les 296 molécules jamais vues : 90.54%

--- Analyse du Vioxx (Rofecoxib) ---
Prédiction IA : NON TOXIQUE (Sécurité estimée à 99.91%)


In [20]:
# 8. TEST SUR LE CYANURE D'HYDROGÈNE (C#N)
mon_smiles_cyanure = "CC(=O)NC1=CC=C(O)C=C1N"

mol_cyanure = Chem.MolFromSmiles(mon_smiles_cyanure)

if mol_cyanure is None:
    print("Erreur : Ce code SMILES est invalide !")
else:
    fp_cyanure = mfpgen.GetFingerprint(mol_cyanure)
    X_cyanure = np.array(fp_cyanure).reshape(1, -1)
    
    probabilite = model.predict(X_cyanure, verbose=0)[0][0]
    
    print(f"--- Analyse du Cyanure d'Hydrogène ({mon_smiles_cyanure}) ---")
    if probabilite >= 0.5:
        print(f"Prédiction IA : TOXIQUE (Risque estimé : {probabilite*100:.2f}%)")
    else:
        print(f"Prédiction IA : NON TOXIQUE (Sécurité estimée : {(1-probabilite)*100:.2f}%)")

--- Analyse du Doliprane (CC(=O)NC1=CC=C(O)C=C1) ---
Prédiction IA : NON TOXIQUE (Sécurité estimée : 79.93%)


In [23]:
# 8. TEST SUR La Thalidomide (Tristement célèbre dans les années 50/60
#  pour avoir causé de graves malformations chez les nouveau-nés). 
#
mon_smiles_mol = "O=C1NC(=O)CCC1N2C(=O)C3=CC=CC=C3C2=O"

mol_pharm = Chem.MolFromSmiles(mon_smiles_mol)

if mol_pharm is None:
    print("Erreur : Ce code SMILES est invalide !")
else:
    fp_mol_pharm = mfpgen.GetFingerprint(mol_pharm)
    X_mol_pharm = np.array(fp_mol_pharm).reshape(1, -1)
    
    probabilite = model.predict(X_mol_pharm, verbose=0)[0][0]
    
    print(f"--- Analyse La Thalidomide ({mon_smiles_mol}) ---")
    if probabilite >= 0.5:
        print(f"Prédiction IA : TOXIQUE (Risque estimé : {probabilite*100:.2f}%)")
    else:
        print(f"Prédiction IA : NON TOXIQUE (Sécurité estimée : {(1-probabilite)*100:.2f}%)")

--- Analyse La Thalidomide (O=C1NC(=O)CCC1N2C(=O)C3=CC=CC=C3C2=O) ---
Prédiction IA : TOXIQUE (Risque estimé : 97.88%)


In [24]:
# Le Valdécoxib (Bextra) (De la même famille que le Vioxx, retiré en 2005
# à cause de sa toxicité pour le cœur).

mon_smiles_mol = "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N"

mol_pharm = Chem.MolFromSmiles(mon_smiles_mol)

if mol_pharm is None:
    print("Erreur : Ce code SMILES est invalide !")
else:
    fp_mol_pharm = mfpgen.GetFingerprint(mol_pharm)
    X_mol_pharm = np.array(fp_mol_pharm).reshape(1, -1)
    
    probabilite = model.predict(X_mol_pharm, verbose=0)[0][0]
    
    print(f"--- Analyse da la Valdécoxib ({mon_smiles_mol}) ---")
    if probabilite >= 0.5:
        print(f"Prédiction IA : TOXIQUE (Risque estimé : {probabilite*100:.2f}%)")
    else:
        print(f"Prédiction IA : NON TOXIQUE (Sécurité estimée : {(1-probabilite)*100:.2f}%)")

--- Analyse da la Valdécoxib (CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N) ---
Prédiction IA : NON TOXIQUE (Sécurité estimée : 99.67%)
